# CRISP Video Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp
from scipy import signal
from scipy.stats import kurtosis, skew, mstats

import copy # use df_copy = copy.deepcopy(df) or use: df_copy = df.iloc[0:,0:]
from datetime import datetime, timedelta
import gc # garbage collection, release memory when deleting variables
import glob
import json
import math
import os
import random
import re

pd.set_option('display.max_rows', 1000) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option('display.max_columns', None)
from IPython.core.interactiveshell import InteractiveShell
#InteractiveShell.ast_node_interactivity = 'all' # 'all'|'last'|'last_expr'|'none'. Default: 'last_expr'.
%autosave 180

In [ ]:
def Stepsize_Cal(end):
    if end > 3000000: stepsize = 200000
    elif end > 1500000: stepsize = 100000
    elif end > 750000: stepsize = 50000
    elif end > 300000: stepsize = 20000
    elif end > 150000: stepsize = 10000
    elif end > 75000: stepsize = 5000
    elif end > 30000: stepsize = 2000
    elif end > 15000: stepsize = 1000
    elif end > 7500: stepsize = 500
    elif end > 3000: stepsize = 200
    elif end > 1500: stepsize = 100
    elif end > 750: stepsize = 50
    elif end > 300: stepsize = 20
    elif end > 150: stepsize = 10
    elif end > 75: stepsize = 5
    elif end > 30: stepsize = 2
    else: stepsize = 1
    return stepsize

In [ ]:
SMALL_SIZE = 16
MEDIUM_SIZE = 18
BIG_SIZE = 20
LARGE_SIZE = 24

params = {
    'figure.figsize': (12, 8),
    'font.size': MEDIUM_SIZE,
    'figure.titlesize': LARGE_SIZE,
    'axes.titlesize': BIG_SIZE,
    'axes.labelsize': BIG_SIZE,
    'xtick.labelsize': MEDIUM_SIZE,
    'ytick.labelsize': MEDIUM_SIZE,
    'legend.fontsize': BIG_SIZE,
}
plt.rcParams.update(params)

In [ ]:
def DF_Summary(df):
    return pd.DataFrame({'Data Type':df.dtypes, 'Missing Data':df.isnull().sum(), 'Unique Values':df.nunique(axis=0, dropna=False)})

In [ ]:
def hist_plot(data_plot, bins=20, xmax=None, precision=0, xlabel='', unit=None, title='', filename=''):
    if unit:
        unit = ' (' + unit + ')'
    else:
        unit = ''
    if xmax:
        data_plot = data_plot[data_plot <= xmax]

    fig,ax = plt.subplots()
    ax.grid(axis='y')
    ax.set_axisbelow(True)
    ax.yaxis.grid(color='gray', linestyle='dashed')

    #bins = np.arange(0.0, round(xmax*1.1), step=1)
    ax.hist(data_plot, histtype='bar', bins=bins)
    start, end = ax.get_ylim()
    end = end * 1.2
    stepsize = Stepsize_Cal(end)
    ax.set(ylim = [0, end])
    ax.yaxis.set_ticks(np.arange(0, end, stepsize))
    ax.set_xlabel(xlabel + unit)
    ax.set_ylabel('Count')
    ax.set_title(title, fontweight='bold')
    if xmax:
        ax.set(xlim=[-xmax*0.1, xmax*1.1])
    
    plt.legend(['Min = {1} {0}\nMax = {2} {0}\nMean = {3} {0}\nMedian = {4} {0}\n0.9 Quantile = {5} {0}\n0.95 Quantile = {6} {0}'.
                format(unit,
                       round(np.min(data_plot), precision),
                       round(np.max(data_plot), precision),
                       round(np.mean(data_plot), precision),
                       round(np.median(data_plot), precision),
                       round(np.quantile(data_plot, 0.9), precision),
                       round(np.quantile(data_plot, 0.95), precision))], loc='best')

    plt.savefig('figures/' + filename + '.png', bbox_inches='tight')

In [ ]:
def diff_pd(df1, df2):
    """Identify differences between two pandas DataFrames"""
    assert (df1.columns == df2.columns).all(), \
        "DataFrame column names are different"
    if any(df1.dtypes != df2.dtypes):
        "Data Types are different, trying to convert"
        df2 = df2.astype(df1.dtypes)
    if df1.equals(df2):
        return None
    else:
        # need to account for np.nan == np.nan returning True
        diff_mask = (df1 != df2) & ~(df1.isnull() & df2.isnull())
        ne_stacked = diff_mask.stack()
        changed = ne_stacked[ne_stacked]
        changed.index.names = ['id', 'col']
        difference_locations = np.where(diff_mask)
        changed_from = df1.values[difference_locations]
        changed_to = df2.values[difference_locations]
        return pd.DataFrame({'from': changed_from, 'to': changed_to},
                            index=changed.index)

In [ ]:
def moving_average(f, npoints):
    # n-Point moving average, this window covers the point and its two sides of equal length.
    # The number of N points in moving average must be an odd number.
    f = np.array(f)
    Nsamples = len(f)
    f_ma = copy.deepcopy(f)
    if Nsamples == 0:
        return f_ma
    if npoints <= 0:
        return f_ma
    if npoints % 2 == 0:
        npoints = npoints + 1
    half_length = int((npoints-1)/2)

    for i in range(len(f)):
        if i < half_length and i < Nsamples/2:
            f_ma[i] = sum(f[:(2*i+1)])/(2*i+1)
        elif i < Nsamples - (npoints-1)/2:
            f_ma[i] = sum(f[(i-half_length):(i+half_length+1)])/npoints
        else:
            f_ma[i] = sum(f[(2*i+1-Nsamples):])/(2*Nsamples-2*i-1)

    return f_ma

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    return b, a

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = signal.lfilter(b, a, data, axis=0)
    return y

def asymmetry(s):
    p = s[s > 0]
    p_mean = np.mean(p)
    p_median = np.median(p)
    p_sum = np.sum(p)
    p_std = np.std(p, ddof=1)
    
    n = np.abs(s[s < 0])
    n_mean = np.mean(n)
    n_median = np.median(n)
    n_sum = np.sum(n)
    n_std = np.std(n, ddof=1)
    
    asymmetry = [np.abs(p_mean-n_mean)*2/(p_mean+n_mean),
                 np.abs(p_median-n_median)*2/(p_median+n_median),
                 np.abs(p_sum-n_sum)*2/(p_sum+n_sum),
                 np.abs(p_std-n_std)*2/(p_std+n_std)]
    #asymmetry = np.abs(skew(s))
    return asymmetry

In [ ]:
def keypoints_extraction(folder, all_points):
    file_list = [f for f in glob.glob(folder + '/**', recursive=True) if f.endswith('.json')]
    file_list.sort()
    if len(file_list) == 0:
        print('No Json files in folder ' + folder)
        return None, None
    
    for i in all_points:
        exec('pose' + str(i) + """ = []""")
    for file in file_list:
        with open(file) as f:
            data_json = json.load(f)
            if len(data_json['people']) > 0:
                for i in all_points:
                    exec('X' + str(i) + """ = data_json['people'][0]['pose_keypoints_2d'][i*3]""")
                    exec('Y' + str(i) + """ = data_json['people'][0]['pose_keypoints_2d'][i*3+1]""")
                    exec('C' + str(i) + """ = data_json['people'][0]['pose_keypoints_2d'][i*3+2]""")
                    exec('pose' + str(i) + """.append([eval('X' + str(i)), eval('Y' + str(i)), eval('C' + str(i))])""")
            else:
                for i in all_points:
                    exec('pose' + str(i) + """.append([0, 0, 0])""")

    for i in all_points:
        exec('pose' + str(i) + """ = pd.DataFrame(eval('pose' + str(i)), columns = ['X', 'Y', 'C'])\
            .replace(0, np.nan).interpolate().iloc[::-1].interpolate().iloc[::-1]""")

    pose_list = []
    for i in all_points:
        pose_list.append(moving_average(np.array(eval('pose' + str(i))), 5))
    return pose_list, len(file_list)

def pose_trace(pose_list, pose_points_of_interest):
    return sum(np.array(pose_list)[pose_points_of_interest])/len(pose_points_of_interest)

def pose_scale(pose_list):
    pose12 = np.sqrt(np.sum(((pose_list[1] - pose_list[2])*(pose_list[1] - pose_list[2]))[:, :-1], axis=1))
    pose15 = np.sqrt(np.sum(((pose_list[1] - pose_list[5])*(pose_list[1] - pose_list[5]))[:, :-1], axis=1))
    return pose12 + pose15

In [ ]:
def getEnvelopeModels(aTimeSeries, rejectCloserThan = 0):
    '''Fits models to the upper and lower envelope peaks and troughs.
    
    A peak is defined as a region where the slope transits from positive to negative (i.e. local maximum).
    A trough is defined as a region where the slope transits from negative to positive (i.e. local minimum).
    
    This example uses cubic splines as models.
    
    Parameters:
    
    aTimeSeries:      A 1 dimensional vector (a list-like).
    rejectCloserThan: An integer denoting the least distance between successive peaks / troughs. Or None to keep all.
    '''
    #Prepend the first value of (s) to the interpolating values. This forces the model to use the same starting point for both the upper and lower envelope models.    
    u_x = [0,]
    u_y = [aTimeSeries[0],]
    lastPeak = 0
    
    l_x = [0,]
    l_y = [aTimeSeries[0],]
    lastTrough = 0
    
    #Detect peaks and troughs and mark their location in u_x,u_y,l_x,l_y respectively.
    for k in range(1,len(aTimeSeries)-1):
        #Mark peaks
        if (np.sign(aTimeSeries[k]-aTimeSeries[k-1])==1) and (np.sign(aTimeSeries[k]-aTimeSeries[k+1])==1) and ((k-lastPeak)>rejectCloserThan):
            u_x.append(k)
            u_y.append(aTimeSeries[k])
            lastPeak = k
            
        #Mark troughs
        if (np.sign(aTimeSeries[k]-aTimeSeries[k-1])==-1) and ((np.sign(aTimeSeries[k]-aTimeSeries[k+1]))==-1) and ((k-lastTrough)>rejectCloserThan):
            l_x.append(k)
            l_y.append(aTimeSeries[k])
            lastTrough = k
    
    #Append the last value of (s) to the interpolating values. This forces the model to use the same ending point for both the upper and lower envelope models.    
    u_x.append(len(aTimeSeries)-1)
    u_y.append(aTimeSeries[-1])
    
    l_x.append(len(aTimeSeries)-1)
    l_y.append(aTimeSeries[-1])
    
    #Fit suitable models to the data. Here cubic splines.
    u_p = sp.interpolate.interp1d(u_x, u_y, kind = 'cubic', bounds_error = False, fill_value=0.0)
    l_p = sp.interpolate.interp1d(l_x, l_y, kind = 'cubic', bounds_error = False, fill_value=0.0)
    
    return (u_p,l_p)

In [ ]:
!mkdir figures

## Data Analysis

In [ ]:
df_video = pd.read_excel('Spreadsheet/CRISP_Video_COCOModel.xlsx')

folder = 'ProcessedVideo/Analyzed'

all_points = range(18) # COCO model
pose_points_of_interest = [1, 2, 5] # example for shoulders and neck, please change based on project's need
fps = 30 # frame per second
mobility_threshold = 0.25 # Unit/Sec, Unit = shoulder width
pose_freq_min = 0.5
pose_freq_max = 6
# choose motion definition
motion_definition = 0 # If motion is defined as scaled difference between two keypoints
#motion_definition = 1 # If motion is defined as scaled difference between two timestamps
clip_ratio = 0.6 # cut the beginning of the video

df_video['FPS'] = 30
df_video['StepInterval'] = None
df_video['Speed2D'] = None
df_video['MobilityIndex'] = None

if motion_definition == 0:
    df_video['MotionAsymmetry'] = None
    df_video['MotionAsymmetryBP'] = None
    df_video['PeakFreqPSD'] = None
    df_video['PeakFreqFFT'] = None
    df_video['MaxAmpFFT'] = None
    df_video['MeanAmpFFT'] = None
    df_video['StdAmpFFT'] = None
    df_video['SkewAmpFFT'] = None
    df_video['KurtosisAmpFFT'] = None
else:
    df_video['VelocityAsymmetry_X'] = None
    df_video['VelocityAsymmetry_Y'] = None
    df_video['PeakFreqPSD_X'] = None
    df_video['PeakFreqPSD_Y'] = None
    df_video['MaxAmpFFT_X'] = None
    df_video['MaxAmpFFT_Y'] = None
    df_video['StdAmpFFT_X'] = None
    df_video['StdAmpFFT_Y'] = None
    
for i in range(len(df_video)):
    video = df_video['VideoName'][i]
    json_folder = folder + '/' + video + '_JSON'
    patient_id =  df_video['PT'][i]
    experiment = df_video['CPEVENT'][i]

    pose_list, num_frame = keypoints_extraction(json_folder, all_points)

    fps = round(num_frame/df_video['TotalTime'][i], -1)
    df_video.at[i, 'FPS'] = fps
    sampling_interval = 1/fps # data sampling interval in time
    start_time = df_video['StartTime'][i]
    end_time = df_video['EndTime'][i]
    start_time = end_time - (end_time-start_time)*clip_ratio
    frame = np.array(range(int(start_time*fps), min(num_frame, int(end_time*fps))))
    num_steps = round(df_video['Steps'][i]*clip_ratio).astype(int)

    scale = pose_scale(pose_list)
    pose = pose_trace(pose_list, pose_points_of_interest)
    data_selected = pose[frame, 0:2]
    dim = data_selected.shape[1]

    motion = np.diff(data_selected, axis=0) / np.repeat(scale[frame[1:]],dim,axis=0).reshape(-1,dim)
    #motion = (np.diff(data_selected[1:, :], axis=0) + np.diff(data_selected[:-1, :], axis=0)) / (2*np.repeat(scale[frame[1:-1]],dim,axis=0).reshape(-1,dim))

    ### Key point trajectory
    
    pose_time = round((frame[-1] - frame[0])/fps, 1)
    step_interval = round((frame[-1] - frame[0])/fps/num_steps, 2)
    speed_array = fps*np.sqrt(np.sum(np.square(motion), axis = 1))
    speed = np.mean(speed_array, axis = 0)
    mobility_index = round(np.sum(speed_array > mobility_threshold)/len(speed_array), 2)

    fig,ax = plt.subplots()
    ax.yaxis.grid(True)

    ax.plot(frame/fps, pose[frame, 0], color='blue')
    ax.plot(frame/fps, pose[frame, 1], color='green')
    ax.plot(frame/fps, moving_average(pose[frame, 0], fps*2+1), linestyle='dashed', color='blue')
    ax.plot(frame/fps, moving_average(pose[frame, 1], fps*2+1), linestyle='dashed', color='green')
    #plt.ylim([0.2, 0.6])

    ax.set_xlabel('Recording Time (Second)')
    ax.set_ylabel('Normalized Coordinate')
    ax.set_title(experiment + ', Body Key Point Trajectory', fontweight='bold')
    ax.legend(['X', 'Y'], loc='best')

    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(1.05, 0.95, 'Result:' +
                        '\n' + str(num_steps) + ' steps take ' +
                        str(pose_time) + ' seconds: ' +
                        str(step_interval) + ' Sec/Step' +
                        '\nAverage 2D Speed: ' + str(round(speed, 2)) + ' Unit/Sec' +
                        '\nMobility Index: ' + str(mobility_index) +
                        '\n' +
                        '\nNote:' +
                        '\nNormalized coordinate range: [0, 1]' +
                        '\n2D Speed: 2D Euclidean speed.' +
                        '\nMobility Index: percentage of samples' +
                        '\nwith 2D speed larger than a threshold' +
                        '\nof ' + str(mobility_threshold) + ' Unit/Sec. (1 Unit = Shoulder Width)',
                        transform=ax.transAxes, verticalalignment='top', bbox=props)
    plt.savefig('figures/' + str(patient_id) + '_' + experiment + '_trajectory.png', bbox_inches='tight')
    plt.close()
    
    hist_plot(speed_array, xmax=4, precision=2,
              xlabel='2D Speed', unit='Unit/Sec',
              title=(experiment + ', Distribution of 2D Speed'),
              filename=(str(patient_id) + '_' + experiment + ', Distribution of 2D Speed'))
    plt.close()
    
    if motion_definition == 0:
        # Elevation difference of left and right shoulders in Y direction
        motion = (pose_trace(pose_list, [5]) - pose_trace(pose_list, [2]))[frame, 1] / scale[frame]
        dim = 1

    ### Motion

    if motion_definition == 0:
        motion_asymmetry = mstats.gmean(asymmetry((pose_trace(pose_list, [5]) - pose_trace(pose_list, [2]))[frame, 1] / scale[frame]))
        #motion_asymmetry = asymmetry((pose_trace(pose_list, [5]) - pose_trace(pose_list, [2]))[frame, 1] / scale[frame])[2]
    fig,ax = plt.subplots()
    plt.plot(frame[-len(motion):]/fps, motion)
    ax.set_xlabel('Recording Time (Second)')
    ax.set_ylabel('Normalized Coordinate')
    ax.set_title(experiment + ', Body Motion', fontweight='bold')
    if motion_definition == 1:
        ax.legend(['X', 'Y'], loc='best')
    plt.savefig('figures/' + str(patient_id) + '_' + experiment + '_motion.png', bbox_inches='tight')
    plt.close()
    
    ### Motion (bandpass filtered)

    data_bp = butter_bandpass_filter(motion, 0.5, 2.5, fps, order=4)
    max_bp = np.abs(data_bp.max(axis=0))
    mean_bp = data_bp.mean(axis=0)
    std_bp = data_bp.std(axis=0, ddof=1)
    skew_bp = skew(data_bp, bias=False)
    kurtosis_bp = kurtosis(data_bp, bias=False)
    
    if motion_definition == 0:
        motion_asymmetry_bp = mstats.gmean(asymmetry(data_bp))
        #motion_asymmetry_bp = asymmetry(data_bp)[2]
    else:
        velocity_asymmetry = np.array([mstats.gmean(asymmetry(data_bp[:, 0])),
                                       mstats.gmean(asymmetry(data_bp[:, 1]))])
    
    fig,ax = plt.subplots()
    plt.plot(frame[-len(motion):]/fps, data_bp)
    ax.set_xlabel('Recording Time (Second)')
    ax.set_ylabel('Normalized Coordinate')
    ax.set_title(experiment + ', Body Motion', fontweight='bold')
    if motion_definition == 1:
        ax.legend(['X', 'Y'], loc='best')
    #plt.psd(motion[:, 0])
    plt.savefig('figures/' + str(patient_id) + '_' + experiment + '_motion_bp.png', bbox_inches='tight')
    plt.close()
    
    ### Motion, FFT

    samples_fft = motion.shape[0] # number of data samples in a time series for FFT analysis
    sampling_interval_fft = 1.0/(samples_fft * sampling_interval) # sampling interval in frequency
    num_fft = int(samples_fft/2) + 1
    freq_index = np.arange(int(np.ceil(pose_freq_min/sampling_interval_fft)), int(pose_freq_max/sampling_interval_fft)+1)
    freq = sampling_interval_fft * freq_index

    data_fft = np.fft.fft(motion, axis=0, norm='ortho')[0:num_fft] # normalized by sqrt(samples_fft)
    data_fft_amp = np.abs(data_fft[freq_index])
    max_amp_fft = data_fft_amp.max(axis=0)
    mean_amp_fft = data_fft_amp.mean(axis=0)
    std_amp_fft = data_fft_amp.std(axis=0, ddof=1)
    skew_amp_fft = skew(data_fft_amp, bias=False)
    kurtosis_amp_fft = kurtosis(data_fft_amp, bias=False)
    peak_freq_fft = freq[data_fft_amp.argmax(axis=0)]
    if dim == 1:
        mean_freq_fft = (data_fft_amp*data_fft_amp*freq).sum(axis=0)/(data_fft_amp*data_fft_amp).sum(axis=0)
    else:
        mean_freq_fft = (data_fft_amp*data_fft_amp*np.repeat(freq,dim,axis=0).reshape(-1,dim)).sum(axis=0)/(data_fft_amp*data_fft_amp).sum(axis=0)
    power_fft = (data_fft_amp*data_fft_amp).sum(axis=0)/len(freq)
    smoothness_fft = np.mean(np.abs(data_fft_amp), axis=0)/np.mean(np.abs(np.diff(data_fft_amp, axis=0)), axis=0)
    #a = data_fft_amp[:, 0]
    #envelope = getEnvelopeModels(a)
    #envelope_u = np.array(list(map(envelope[0],range(0,len(a)))))
    #envelope_l = np.array(list(map(envelope[1],range(0,len(a)))))
    #np.mean(envelope_u - envelope_l)/np.mean(a)
    #fluctuation_x = (len(signal.find_peaks(a)[0]) + len(signal.find_peaks(a)[0]))/fps

    fig,ax = plt.subplots()
    plt.plot(freq, data_fft_amp)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')
    ax.set_title(experiment + ', Distribution of Motion Frequency', fontweight='bold')
    if motion_definition == 1:
        ax.legend(['X, Peak frequency: {} Hz'.format(round(peak_freq_fft[0], 2)),
                   'Y, Peak frequency: {} Hz'.format(round(peak_freq_fft[1], 2))], loc='best')
    plt.savefig('figures/' + str(patient_id) + '_' + experiment + '_frequency_distribution.png', bbox_inches='tight')
    plt.close()
    
    ### Motion, PSD

    f, PSD = signal.welch(motion, fps, nfft=len(motion), axis=0)
    sampling_interval_fft = np.mean(np.diff(f))
    freq_index = np.arange(int(np.ceil(pose_freq_min/sampling_interval_fft)), int(pose_freq_max/sampling_interval_fft)+1)
    freq = f[freq_index]

    data_PSD = PSD[freq_index]
    max_PSD = data_PSD.max(axis=0)
    mean_PSD = data_PSD.mean(axis=0)
    std_PSD = data_PSD.std(axis=0, ddof=1)
    skew_PSD = skew(data_PSD, bias=False)
    kurtosis_PSD = kurtosis(data_PSD, bias=False)
    peak_freq_PSD = freq[data_PSD.argmax(axis=0)]
    if dim == 1:
        mean_freq_PSD = (data_PSD*freq).sum(axis=0)/data_PSD.sum(axis=0)
    else:
        mean_freq_PSD = (data_PSD*np.repeat(freq,dim,axis=0).reshape(-1,dim)).sum(axis=0)/data_PSD.sum(axis=0)

    fig,ax = plt.subplots()
    plt.plot(f[freq_index], PSD[freq_index])
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power Spectral Density')
    ax.set_title(experiment + ', Distribution of Power Spectral Density', fontweight='bold')
    if motion_definition == 1:
        ax.legend(['X, Peak frequency: {} Hz'.format(round(peak_freq_PSD[0], 2)),
                   'Y, Peak frequency: {} Hz'.format(round(peak_freq_PSD[1], 2))], loc='best')
    plt.savefig('figures/' + str(patient_id) + '_' + experiment + '_PSD.png', bbox_inches='tight')
    plt.close()
    
    df_video.at[i, 'StepInterval'] = round(step_interval, 2)
    df_video.at[i, 'Speed2D'] = round(speed, 2)
    df_video.at[i, 'MobilityIndex'] = round(mobility_index, 2)
    
    if motion_definition == 0:
        df_video.at[i, 'MotionAsymmetry'] = round(motion_asymmetry, 3)
        df_video.at[i, 'MotionAsymmetryBP'] = round(motion_asymmetry_bp, 3)
        df_video.at[i, 'PeakFreqPSD'] = round(peak_freq_PSD, 3)
        df_video.at[i, 'PeakFreqFFT'] = round(peak_freq_fft, 3)
        df_video.at[i, 'MaxAmpFFT'] = round(max_amp_fft, 3)
        df_video.at[i, 'MeanAmpFFT'] = round(mean_amp_fft, 3)
        df_video.at[i, 'StdAmpFFT'] = round(std_amp_fft, 3)
        df_video.at[i, 'SkewAmpFFT'] = round(skew_amp_fft, 3)
        df_video.at[i, 'KurtosisAmpFFT'] = round(kurtosis_amp_fft, 3)
    else:
        df_video.at[i, 'VelocityAsymmetry_X'] = round(velocity_asymmetry[0], 3)
        df_video.at[i, 'VelocityAsymmetry_Y'] = round(velocity_asymmetry[1], 3)
        df_video.at[i, 'PeakFreqPSD_X'] = round(peak_freq_PSD[0], 3)
        df_video.at[i, 'PeakFreqPSD_Y'] = round(peak_freq_PSD[1], 3)
        df_video.at[i, 'MaxAmpFFT_X'] = round(max_amp_fft[0], 3)
        df_video.at[i, 'MaxAmpFFT_Y'] = round(max_amp_fft[1], 3)
        df_video.at[i, 'StdAmpFFT_X'] = round(std_amp_fft[0], 3)
        df_video.at[i, 'StdAmpFFT_Y'] = round(std_amp_fft[1], 3)

In [ ]:
df_video.to_excel('./Spreadsheet/CRISP_Video_Feature_Output.xlsx', index=False)